# Simple Reduce Kernel with GPU

In [1]:
import torch
import triton
from torch import Tensor
import triton.language as tl
import jaxtyping
from jaxtyping import *

In [23]:
N = 32768

In [24]:
X = torch.arange(N, device='cuda')

In [25]:
@triton.jit
def reduce_kernel(X, Y, N, BLOCK_SIZE : tl.constexpr):
    pid = tl.program_id(axis=0)

    data = tl.load(X + pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE))
    val = tl.sum(data)
    tl.store(Y + pid, val)

In [35]:
import math

def reduce(X_in):
    """
    Assumes perfect power of reduce_factor
    """
    num_elems = N
    reduce_factor = 32
    X = torch.clone(X_in)
    Y = torch.empty((N // reduce_factor,), device='cuda')

    while num_elems > 1:
        reduce_kernel[(math.ceil(num_elems / reduce_factor),)]\
            (X, Y, num_elems, reduce_factor)
        num_elems //= reduce_factor
        X, Y = Y, X

    return X[0].item()

In [36]:
reduce(X)

536854528.0

In [37]:
torch.sum(X)

tensor(536854528, device='cuda:0')